# 🖱️ Phase 3 — Forms, Login & Interaction
### PixelCraft Infotech | Web Scraping Course

**What you'll learn:**
- Type into text fields with `send_keys`
- Submit forms and handle login flows
- Use `Select` for dropdown menus
- Hover and drag with `ActionChains`
- Scroll pages with `execute_script`
- Handle infinite scroll

---

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import time
import csv

options = Options()
options.add_argument("--headless=new")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")

print("✅ Imports ready")

✅ Imports ready


---
## Step 1 — send_keys: Typing into Input Fields

> `element.send_keys("text")` types text into an input field  
> `Keys.ENTER` simulates pressing Enter  
> `element.clear()` clears existing text before typing

In [2]:
# We'll use quotes.toscrape.com search via URL trick
# For send_keys demo — using the login page

driver = webdriver.Chrome(options=options)
driver.get("http://quotes.toscrape.com/login")

wait = WebDriverWait(driver, 10)

# Wait for the username field
username_field = wait.until(
    EC.presence_of_element_located((By.ID, "username"))
)

password_field = driver.find_element(By.ID, "password")

# Clear any existing text
username_field.clear()
password_field.clear()

# Type into fields
username_field.send_keys("admin")
password_field.send_keys("admin")

print("✅ Typed credentials")
print("Username field value:", username_field.get_attribute("value"))

driver.quit()

✅ Typed credentials
Username field value: admin


---
## Step 2 — Login Flow

> After typing credentials → click the submit button  
> Verify login success by checking URL or a logged-in element

In [3]:
driver = webdriver.Chrome(options=options)
driver.get("http://quotes.toscrape.com/login")

wait = WebDriverWait(driver, 10)

# Fill login form
wait.until(EC.presence_of_element_located((By.ID, "username"))).send_keys("admin")
driver.find_element(By.ID, "password").send_keys("admin")

# Click the Login button
login_btn = driver.find_element(By.CSS_SELECTOR, "input[type='submit']")
login_btn.click()

# Wait for redirect after login
wait.until(EC.url_contains("/"))

print("Current URL:", driver.current_url)

# Verify logged in — look for 'Logout' link
try:
    logout_link = driver.find_element(By.LINK_TEXT, "Logout")
    print("✅ Login successful — Logout link found!")
except NoSuchElementException:
    print("❌ Login failed")

driver.quit()

Current URL: https://quotes.toscrape.com/login
✅ Login successful — Logout link found!


---
## Step 3 — Login → Scrape (Session Maintained)

> After login, Selenium keeps the session (cookies) alive.  
> You can scrape pages that require authentication.

In [4]:
driver = webdriver.Chrome(options=options)
driver.get("http://quotes.toscrape.com/login")

wait = WebDriverWait(driver, 10)

# Login
wait.until(EC.presence_of_element_located((By.ID, "username"))).send_keys("admin")
driver.find_element(By.ID, "password").send_keys("admin")
driver.find_element(By.CSS_SELECTOR, "input[type='submit']").click()
wait.until(EC.presence_of_element_located((By.CLASS_NAME, "quote")))

print("✅ Logged in — now scraping authenticated page")
print("URL:", driver.current_url)

# Scrape first page while logged in
quotes = driver.find_elements(By.CLASS_NAME, "quote")
print(f"Found {len(quotes)} quotes while authenticated")

# Print cookies (session is maintained)
cookies = driver.get_cookies()
print(f"\nActive cookies: {len(cookies)}")
for c in cookies:
    print(f"  {c['name']} = {c['value'][:30]}...")

driver.quit()

✅ Logged in — now scraping authenticated page
URL: https://quotes.toscrape.com/
Found 10 quotes while authenticated

Active cookies: 1
  session = eyJjc3JmX3Rva2VuIjoiSmFqWnhORX...


---
## Step 4 — Select Dropdowns

> `Select(element)` wraps a `<select>` dropdown  
> `.select_by_visible_text()` — choose by label  
> `.select_by_value()` — choose by HTML value  
> `.select_by_index()` — choose by position (0-based)

We'll demo with a test site: `https://the-internet.herokuapp.com/dropdown`

In [5]:
driver = webdriver.Chrome(options=options)
driver.get("https://the-internet.herokuapp.com/dropdown")

wait = WebDriverWait(driver, 10)
dropdown_el = wait.until(EC.presence_of_element_located((By.ID, "dropdown")))

# Wrap with Select class
dropdown = Select(dropdown_el)

# Print all options
print("Available options:")
for option in dropdown.options:
    print(f"  value='{option.get_attribute('value')}' → '{option.text}'")

# Select Option 1 by visible text
dropdown.select_by_visible_text("Option 1")
print("\n✅ Selected:", dropdown.first_selected_option.text)

# Select Option 2 by value
dropdown.select_by_value("2")
print("✅ Selected:", dropdown.first_selected_option.text)

driver.quit()

Available options:
  value='' → 'Please select an option'
  value='1' → 'Option 1'
  value='2' → 'Option 2'

✅ Selected: Option 1
✅ Selected: Option 2


---
## Step 5 — ActionChains (Hover & Keyboard Shortcuts)

> `ActionChains` lets you chain complex user interactions  
> Useful for: hover menus, drag-and-drop, keyboard shortcuts

In [6]:
driver = webdriver.Chrome(options=options)
driver.get("https://the-internet.herokuapp.com/hovers")

wait = WebDriverWait(driver, 10)

# Find all avatar images to hover over
avatars = wait.until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".figure img"))
)

actions = ActionChains(driver)

for i, avatar in enumerate(avatars):
    # Hover over avatar
    actions.move_to_element(avatar).perform()
    time.sleep(0.5)  # brief pause for DOM update

    # Try to read the caption that appears on hover
    try:
        caption = driver.find_elements(By.CSS_SELECTOR, ".figure .figcaption h5")[i]
        print(f"Avatar {i+1} hover text: {caption.text}")
    except Exception as e:
        print(f"Avatar {i+1}: could not read caption")

driver.quit()

Avatar 1 hover text: name: user1
Avatar 2 hover text: name: user2
Avatar 3 hover text: name: user3


---
## Step 6 — Scrolling with execute_script

> `driver.execute_script(js_code)` runs any JavaScript  
> Used for scrolling, clicking hidden elements, getting page info

In [ ]:
driver = webdriver.Chrome(options=options)
driver.get("http://quotes.toscrape.com/scroll")  # infinite scroll version!

wait = WebDriverWait(driver, 10)
wait.until(EC.presence_of_element_located((By.CLASS_NAME, "quote")))

all_quotes = []
last_count = 0
scroll_attempts = 0
max_scrolls = 15

while scroll_attempts < max_scrolls:
    # Scroll to the bottom of the page
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(1.5)   # give time for new content to load

    # Count quotes now
    current_quotes = driver.find_elements(By.CLASS_NAME, "quote")
    current_count = len(current_quotes)

    print(f"Scroll {scroll_attempts+1}: {current_count} quotes loaded")

    # If no new quotes loaded → we've reached the end
    if current_count == last_count:
        print("✅ No new quotes — infinite scroll complete!")
        break

    last_count = current_count
    scroll_attempts += 1

# Now collect all data
for q in driver.find_elements(By.CLASS_NAME, "quote"):
    try:
        text   = q.find_element(By.CLASS_NAME, "text").text.strip()
        author = q.find_element(By.CLASS_NAME, "author").text.strip()
        all_quotes.append({"quote": text, "author": author})
    except Exception:
        pass

driver.quit()
print(f"\n✅ Total quotes from infinite scroll: {len(all_quotes)}")

In [ ]:
# More execute_script tricks

driver = webdriver.Chrome(options=options)
driver.get("http://quotes.toscrape.com")

wait = WebDriverWait(driver, 10)
wait.until(EC.presence_of_element_located((By.CLASS_NAME, "quote")))

# Scroll to a specific element
element = driver.find_element(By.CSS_SELECTOR, "li.next a")
driver.execute_script("arguments[0].scrollIntoView(true);", element)
print("✅ Scrolled to Next button")

# Click hidden element (useful when element is behind overlay)
driver.execute_script("arguments[0].click();", element)
print("✅ Clicked via JS — URL:", driver.current_url)

# Get page dimensions
height = driver.execute_script("return document.body.scrollHeight")
print(f"Page height: {height}px")

driver.quit()

---
## Step 7 — Searching with Keyboard Input

> Many sites have search bars. Let's type → press Enter → scrape results.

In [ ]:
# Demo: Author search on quotes.toscrape.com
# The site uses /author/author-name URLs, so we'll demo the pattern

driver = webdriver.Chrome(options=options)

# Navigate directly to author page (simulating a search)
authors_to_check = ["Albert-Einstein", "J-K-Rowling", "Jane-Austen"]
author_data = []

for author_slug in authors_to_check:
    driver.get(f"http://quotes.toscrape.com/author/{author_slug}/")

    wait = WebDriverWait(driver, 10)
    try:
        name = wait.until(
            EC.presence_of_element_located((By.CLASS_NAME, "author-title"))
        ).text
        born = driver.find_element(By.CLASS_NAME, "author-born-date").text
        desc = driver.find_element(By.CLASS_NAME, "author-description").text[:100]

        author_data.append({"name": name, "born": born, "description": desc})
        print(f"✅ {name} — Born: {born}")

    except TimeoutException:
        print(f"❌ Could not find author: {author_slug}")

driver.quit()
print(f"\nScraped {len(author_data)} author profiles")

---
## 🏆 Class Exercise

1. Go to `http://quotes.toscrape.com/login`, log in with `admin/admin`
2. After login, scrape ALL quotes across all pages (you're now authenticated)
3. Save them to `authenticated_quotes.csv`
4. Go to `http://quotes.toscrape.com/scroll` and use infinite scroll to get ALL quotes
5. Compare: did you get the same 100 quotes from both methods?

**Bonus:** Use `execute_script` to highlight each quote element in yellow before scraping it  
*(hint: `arguments[0].style.backgroundColor = 'yellow'`)*

In [ ]:
# Bonus: Highlight elements as you scrape
def highlight(driver, element):
    """Flash an element yellow before scraping it — great for debugging."""
    driver.execute_script("arguments[0].style.backgroundColor = 'yellow';", element)
    time.sleep(0.1)
    driver.execute_script("arguments[0].style.backgroundColor = '';", element)

print("✅ highlight() function ready — call highlight(driver, element) before each scrape")